In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from itpo_weights import DatasetType


In [ ]:
# Load the results
dataset_type = DatasetType.NodeOptimized # Match this to your actual enum string
main_directory = os.path.join("./cross_val", f"{dataset_type}", "MST")
results_path = os.path.join(main_directory, "k_fold_results.pkl")
fold_results = torch.load(results_path)

fig, ax = plt.subplots(1, 3, figsize=(9.99, 3.33), layout="tight", dpi=300)

validation_interval = 5
epochs_total = 150 
x_axis = np.arange(0, epochs_total, validation_interval)


# Plot individual folds
all_fold_mses = []
all_fold_r2s = []
for i, fold in enumerate(fold_results):

    # Plot training loss
    training_loss_curve = fold['training_loss']
    ax[0].plot(range(epochs_total), training_loss_curve, label=f"Fold {i+1}")

    # Plot rollout r2
    r2_curve = fold['r2']
    ax[1].plot(x_axis, r2_curve, label=f'Fold {i+1}', linewidth=1.5)
    all_fold_r2s.append(r2_curve)

    # Depending on how MSEs were saved, you might need to average the rollout 
    # steps if 'rollout_mse' is a list of lists.
    # Assuming fold['rollout_mse'] is a 1D list of scalar values per validation step:
    val_mse_curve = [np.mean(step_mses) for step_mses in fold['rollout_mse']] 
    all_fold_mses.append(val_mse_curve)
    ax[2].plot(x_axis, val_mse_curve, label=f'Fold {i+1}', alpha=0.4, linewidth=1.5)

# ax[0].set_title(f"Training loss over {epochs_total} Epochs")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Acceleration MSE")
ax[0].set_yscale('log')
# ax[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax[0].grid(True, which="both", ls="--", alpha=0.5)

# ax[1].set_title(f"Rollout $R^2$ over {epochs_total} Epochs")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("$R^2$")
ax[1].set_yticks(np.linspace(0, 1.0, 11))
# ax[1].set_yscale('log')
# ax[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax[1].grid(True, which="both", ls="--", alpha=0.5)
ax[1].set_ylim(0, 1.05)


# Calculate and plot the mean curve
all_fold_mses = np.array(all_fold_mses)
mean_mse = np.mean(all_fold_mses, axis=0)

ax[2].plot(x_axis, mean_mse, color='black', label='Mean', linewidth=3, linestyle='--')

# Formatting
# ax[2].set_title(f"Validation Position MSE over {epochs_total} Epochs")
ax[2].set_xlabel("Epoch")
ax[2].set_ylabel("Position MSE")
ax[2].set_yscale('log') # Usually best to view MSE on a log scale
ax[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax[2].grid(True, which="both", ls="--", alpha=0.5)

# Save and show
# plot_path = os.path.join(main_directory, "cv_validation_curves.png")
# plt.savefig(plot_path, dpi=300)
# print(f"Plot saved to {plot_path}")
plt.show()

### Test on lefover test data

#### Load the models

In [ ]:
from torch_geometric.data import Data

from simulator_SA_cpu_test import Model as VelocityModel
from training_utils import freeze_normalizer

mp_layers = 2
mlp = 3
hidden_size = 128
history = 3
device = "cuda"

init_graph = Data(x=torch.ones((100, history*2)), edge_attr=torch.ones((100, 4)))

lucky_models = {}

model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
model.load_checkpoint("./good_models/f4_e50.pt")
model = freeze_normalizer(model)
lucky_models[f"h{3} f4 e50"] = model

# model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
# model.load_checkpoint("./good_models/f5_e55.pt")
# model = freeze_normalizer(model)
# lucky_models[f"h{3} f5 e55"] = model

model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
model.load_checkpoint("./good_models/f5_e55.pt")
model = freeze_normalizer(model)
lucky_models[f"h{3} f10 e120"] = model


for i, model_name in enumerate(lucky_models.keys()):
    print(f"{i+1}. Model {model_name}.")

#### Get unused data

In [ ]:
import pandas as pd
from tqdm import tqdm

from graph_utils import prepare_traj


def get_sampled_files_with_labels(
    registry_path: str,
    target_data_type: DatasetType,
    poisson_buckets: list,
    seed: int = 42,
) -> tuple[list[str], list[int]]:
    """
    Samples data files and returns a tuple of:
    1. A shuffled list of file paths.
    2. A matching list of bucket integer IDs to use for stratification.
    """
    df = pd.read_csv(registry_path)
    type_df = df[df["data_type"] == str(target_data_type)]

    if type_df.empty:
        raise ValueError(f"No data found for data_type: {target_data_type!s}")

    sampled_dfs = []

    for i, bucket in enumerate(poisson_buckets):
        p_min = bucket.get("min", float("-inf"))
        p_max = bucket.get("max", float("inf"))
        req_count = bucket["count"]

        bucket_df = type_df[
            (type_df["poisson_ratio"] >= p_min) & (type_df["poisson_ratio"] < p_max)
        ].copy()

        available = len(bucket_df)
        if available == 0:
            print(f"Bucket {i} ({p_min} <= P < {p_max}) is empty.")
            continue

        req_count = min(req_count, available)

        sampled = bucket_df.sample(n=req_count, random_state=seed)

        # Tag each row with its specific bucket index for stratification
        sampled["bucket_id"] = i
        sampled_dfs.append(sampled)

    if not sampled_dfs:
        raise ValueError("No data was sampled from any bucket. Check threshold logic.")

    # Combine and shuffle while maintaining the index match between files and bucket_ids
    combined_df = (pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True))

    return combined_df["file_path"].tolist(), combined_df["bucket_id"].tolist()


def get_unsampled_holdout_files(
    registry_path: str,
    target_data_type: DatasetType,
    poisson_buckets: list,
    seed: int = 42
) -> list[str]:
    """
    Identifies all files in the registry that were NOT selected 
    during the initial Stratified K-Fold sampling step.
    """
    # 1. Re-run the exact same sampling to get the "used" files
    used_files, _ = get_sampled_files_with_labels(
        registry_path=registry_path,
        target_data_type=target_data_type,
        poisson_buckets=poisson_buckets,
        seed=seed
    )
    
    # 2. Load the full registry and filter by your target data type
    df = pd.read_csv(registry_path)
    type_df = df[df['data_type'] == str(target_data_type)]
    
    # 3. Create a boolean mask to filter out anything that was used
    used_set = set(used_files)
    holdout_df = type_df[~type_df['file_path'].isin(used_set)]
    
    leftover_files = holdout_df['file_path'].tolist()
    
    print(f"Total files of this type: {len(type_df)}")
    print(f"Files used in CV: {len(used_set)}")
    print(f"Unseen holdout files extracted: {len(leftover_files)}")
    
    return leftover_files

poisson_buckets = [
    {"max": 0.1, "count": 500},  # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 500},  # 0.1 <= P < 0.2
    {"min": 0.2, "count": 300},  # P >= 0.2
]

unused_sims = get_unsampled_holdout_files(
    "./data/data_registry.csv",
    DatasetType.NodeOptimized,
    poisson_buckets,
    seed=42
)

max_sim_len = 100
test_data = []
for file in tqdm(unused_sims, desc="Loading & Preparing Data"):
    sim = torch.load(file, weights_only=False)[:max_sim_len]
    prepared_sim = prepare_traj(sim, calc_angles=False)
    test_data.append(prepared_sim)

In [ ]:
from copy import deepcopy

import barostat_parameters
from utils import calc_p_ratio_box_tensor, get_rollout

factors = [(sim[4].box_tensor[0]/sim[3].box_tensor[0]).item() for sim in test_data]
mean_factor = sum(factors)/len(factors)

num_steps = 50
model_results = {}

if dataset_type is DatasetType.NodeOptimized:
    barostat_config = barostat_parameters.node_optimizated
elif dataset_type is DatasetType.StiffOptimized:
    barostat_config = barostat_parameters.stiff_optimized
elif dataset_type is DatasetType.Noisy:
    print("Using noise data and corresponding barostat parameters:")
    barostat_config = barostat_parameters.noisy
    print(barostat_config)

name_len = max([len(n) for n in lucky_models])+1 

for model_name, model in lucky_models.items():
    model.eval()

    history = int(model_name[1])
    target_idx = num_steps + history + 1
    results = {
        "gt_box_velocity": [],
        "pred_box_velocity": [],
        "final_mse": [],
        "mse": [],
        "est_p": [],
        "pred_p": [],
        "gt_est_p": [],
        "gt_p": [],
        "gt_box": [],
        "pred_box": [],
        "gt_forces": [],
        "pred_forces": [],
        "gt_pressure": [],
        "pred_pressure": [],
    }

    with torch.no_grad():
        for test_sim in tqdm(test_data, desc=f"Model {model_name:<{name_len}}"):

            # This block computes dumping period (N MD steps in 1 dump step)        
            sim_strain = (test_sim[1].box.x - test_sim[-1].box.x) / test_sim[0].box.x
            assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
            dump_period = int(assumed_rollout_length / len(test_sim)) + 1

            barostat_config_current = deepcopy(barostat_config)
            barostat_config_current["default_skip"] = dump_period

            input_graphs = [g.cpu().detach() for g in test_sim[: history + 1]]

            rollout = get_rollout(
                input_graphs=input_graphs,
                gnn_simulator=model,
                gnn_history=history,
                num_steps=num_steps,
                barostat_config=barostat_config_current,
                device="cuda"
            )

            # Compare Predicted Position vs Ground Truth Position
            pos_mse = torch.nn.functional.mse_loss(rollout[-1].pos.cpu(), test_sim[target_idx].to(device).pos.cpu())
            results["final_mse"].append(pos_mse.item())
            pos_mse = [torch.nn.functional.mse_loss(rollout[i].pos.cpu(), test_sim[i].to(device).pos.cpu()).item() for i in range(len(rollout))]
            results["mse"].append(pos_mse)
            
            pred_p = calc_p_ratio_box_tensor(rollout).item()
            results["pred_p"].append(pred_p)
            gt_p = calc_p_ratio_box_tensor(test_sim[:target_idx]).item()
            results["gt_p"].append(gt_p)

            pred_box = [g.box_tensor.cpu().detach() for g in rollout]
            results["pred_box"].append(pred_box)
            gt_box = [g.box_tensor.cpu().detach() for g in test_sim[: len(rollout)]]
            results["gt_box"].append(gt_box)

            # gt_pressure = torch.stack([compute_total_stress(g, r0=input_graphs[0].edge_attr[:, -2].to(g.x.device), temperature=1e-7).cpu() for g in test_sim[: len(rollout)]], dim=0)
            # results['gt_pressure'].append(gt_pressure)
            # pred_pressure = torch.stack([compute_total_stress(g, r0=input_graphs[0].edge_attr[:, -2].to(g.x.device), temperature=1e-7).cpu() for g in rollout], dim=0)
            # results["pred_pressure"].append(pred_pressure)

    model_results[model_name] = results

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

fig, ax = plt.subplots(1, 1, layout="constrained")

line = (min(results["gt_p"]) - 0.05, max(results["gt_p"]) + 0.05)
ax.plot(line, line, color="black", linewidth=1, linestyle='--')

for model_name, results in model_results.items():

    r2 = r2_score(results['gt_p'], results['pred_p'])
    res = spearmanr(results["gt_p"], results["pred_p"])
    sp = res.statistic
    
    label_text = f"Model {model_name}: $R^2={r2:.3f}$, SP={sp:.3f}"
    ax.scatter(results["gt_p"], results["pred_p"], label=label_text)
    
ax.legend(frameon=False, loc='best')
ax.set_xlabel(r"$\nu_{gt}$")
ax.set_ylabel(r"$\nu_{pred}$")
# ax.set_ylim(-0.6, 0.6)


plt.show()

### A different way of picking data

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm


def get_cv_and_test_pools(
    registry_path: str,
    target_data_type: str,
    cv_buckets: list,
    test_buckets: list,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Safely extracts a Holdout Test Pool, then extracts a Cross-Validation Pool 
    from the REMAINING data to guarantee absolutely zero data leakage.
    """
    df = pd.read_csv(registry_path)
    type_df = df[df["data_type"] == str(target_data_type)].copy()

    if type_df.empty:
        raise ValueError(f"No data found for data_type: {target_data_type}")

    def sample_buckets(pool_df, buckets, pool_name):
        sampled_dfs = []
        for i, bucket in enumerate(buckets):
            p_min = bucket.get("min", float("-inf"))
            p_max = bucket.get("max", float("inf"))
            req_count = bucket["count"]

            bucket_df = pool_df[(pool_df["poisson_ratio"] >= p_min) & (pool_df["poisson_ratio"] < p_max)].copy()
            
            available = len(bucket_df)
            if available == 0:
                logging.warning(f"{pool_name} Bucket {i} ({p_min} <= P < {p_max}) is empty.")  # noqa: LOG015
                continue

            take_count = min(req_count, available)
            sampled = bucket_df.sample(n=take_count, random_state=seed)
            sampled["bucket_id"] = i
            sampled_dfs.append(sampled)
            
        if not sampled_dfs:
            raise ValueError(f"No data sampled for {pool_name}.")
            
        return pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True)

    # 1. Sample the Test pool first
    test_df = sample_buckets(type_df, test_buckets, "Test")
    
    # 2. Remove Test data from available pool
    remaining_df = type_df[~type_df["file_path"].isin(test_df["file_path"])]
    
    # 3. Sample CV pool from the leftovers
    cv_df = sample_buckets(remaining_df, cv_buckets, "CV")

    return cv_df, test_df


def plot_fold_distributions(train_df, val_df, test_df, fold_num, save_dir):
    """Generates a 3-panel histogram visualizing the exact distributions used in the fold."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    
    bins = np.linspace(-0.5, 0.5, 20) # Adjust based on your actual physical P limits
    
    # Plot Train
    axes[0].hist(train_df['poisson_ratio'], bins=bins, color='skyblue', edgecolor='black')
    axes[0].set_title(f"Train Set (n={len(train_df)})")
    axes[0].set_xlabel("Poisson's Ratio")
    axes[0].set_ylabel("Count")
    
    # Plot Validation
    axes[1].hist(val_df['poisson_ratio'], bins=bins, color='lightgreen', edgecolor='black')
    axes[1].set_title(f"Validation Set (n={len(val_df)})")
    axes[1].set_xlabel("Poisson's Ratio")
    
    # Plot Test
    axes[2].hist(test_df['poisson_ratio'], bins=bins, color='salmon', edgecolor='black')
    axes[2].set_title(f"Test Set (n={len(test_df)})")
    axes[2].set_xlabel("Poisson's Ratio")
    
    fig.suptitle(f"Data Distribution for Fold {fold_num}", fontsize=14)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f"fold_{fold_num}_distributions.png")
    plt.savefig(plot_path, dpi=200)
    plt.close()
    logging.info(f"Saved distribution histogram to {plot_path}")  # noqa: LOG015


cv_buckets = [
    {"max": 0.1, "count": 500},  
    {"min": 0.1, "max": 0.2, "count": 500},  
    {"min": 0.2, "count": 300},  
]

test_buckets = [
    {"max": 0.1, "count": 70},  
    {"min": 0.1, "max": 0.2, "count": 70},  
    {"min": 0.2, "count": 70},  
]

cv_pool, test_pool = get_cv_and_test_pools(
    registry_path="./data/data_registry.csv",
    target_data_type=DatasetType.NodeOptimized,
    cv_buckets=cv_buckets,
    test_buckets=test_buckets,
    seed=42
)


In [21]:
# Optional: Load ALL data into memory ONCE based on these DataFrames
# master_dataset = load_your_tensors_here(cv_df['file_path'].tolist() + test_df['file_path'].tolist())

k_folds = 3
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# ==========================================
# EXPERIMENT CONFIGURATION
# Adjust these variables to test your hypothesis!
# ==========================================
# TRAIN_P_MIN = 0.1
TRAIN_P_MIN = float('-inf')
TRAIN_P_MAX = float('inf')

# VAL_P_MIN = 0.1  # Set this to -inf to validate on full range, or 0.1 to validate on high P
VAL_P_MIN = float('-inf')  
VAL_P_MAX = float('inf')

for fold, (train_idx, val_idx) in enumerate(skf.split(cv_pool, cv_pool["bucket_id"])):
    print(f"\n{'='*40}\nStarting Fold {fold + 1}/{k_folds}\n{'='*40}")
    
    # 1. Get the mathematically split raw data for this fold
    raw_train_df = cv_pool.iloc[train_idx]
    raw_val_df = cv_pool.iloc[val_idx]

    # 2. DYNAMICALLY FILTER THE DISTRIBUTIONS
    fold_train_df = raw_train_df[
        (raw_train_df['poisson_ratio'] >= TRAIN_P_MIN) & 
        (raw_train_df['poisson_ratio'] < TRAIN_P_MAX)
    ]
    
    fold_val_df = raw_val_df[
        (raw_val_df['poisson_ratio'] >= VAL_P_MIN) & 
        (raw_val_df['poisson_ratio'] < VAL_P_MAX)
    ]
    
    # 3. Visualize the distributions before doing any training
    main_directory = "./data_splits"
    os.makedirs(main_directory, exist_ok=True)
    fold_save_dir = os.path.join(main_directory, f"fold_{fold + 1}")
    os.makedirs(fold_save_dir, exist_ok=True)
    
    plot_fold_distributions(
        train_df=fold_train_df, 
        val_df=fold_val_df, 
        test_df=test_pool, 
        fold_num=fold + 1, 
        save_dir=fold_save_dir
    )


Starting Fold 1/3

Starting Fold 2/3

Starting Fold 3/3


### Yet another way of splitting nicely

In [ ]:
import pandas as pd


def get_explicit_splits(
    registry_path: str,
    target_data_type: str,
    train_config: dict,
    val_config: dict,
    seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Explicitly draws N training samples and M validation samples based on predefined P-ranges.
    Everything left over defaults to the Test set.
    """
    df = pd.read_csv(registry_path)
    pool_df = df[df["data_type"] == str(target_data_type)].copy()

    if pool_df.empty:
        raise ValueError(f"No data found for data_type: {target_data_type}")

    # 1. Sample Training Data
    train_candidates = pool_df[
        (pool_df["poisson_ratio"] >= train_config["p_min"]) & 
        (pool_df["poisson_ratio"] < train_config["p_max"])
    ]
    
    if len(train_candidates) < train_config["count"]:
        raise ValueError(f"Train requested {train_config['count']}, but only {len(train_candidates)} available in range.")
        
    train_df = train_candidates.sample(n=train_config["count"], random_state=seed)
    
    # Remove the selected training data from the master pool to prevent leakage
    pool_df = pool_df.drop(train_df.index)

    # 2. Sample Validation Data
    val_candidates = pool_df[
        (pool_df["poisson_ratio"] >= val_config["p_min"]) & 
        (pool_df["poisson_ratio"] < val_config["p_max"])
    ]
    
    if len(val_candidates) < val_config["count"]:
        raise ValueError(f"Val requested {val_config['count']}, but only {len(val_candidates)} available in range.")
        
    val_df = val_candidates.sample(n=val_config["count"], random_state=seed)
    
    # Remove the selected validation data from the master pool
    pool_df = pool_df.drop(val_df.index)

    # 3. The Rest is Testing
    test_df = pool_df.copy()

    return train_df, val_df, test_df

dataset_type = DatasetType.NodeOptimized

# Your explicit requirements!
train_config = {
    "count": 200, 
    "p_min": 0.1, 
    "p_max": float('inf')
}

val_config = {
    "count": 200, 
    "p_min": 0.1, 
    "p_max": float('inf')
}

n_tries = 5 # Number of times you want to run this experiment
fold_results = []

for try_num in range(n_tries):
    seed = 42 + try_num # Change seed to draw a new random subset each time
    # logger.info(f"\n{'='*40}\nStarting Try {try_num + 1}/{n_tries}\n{'='*40}")
    print(f"\n{'='*40}\nStarting Try {try_num + 1}/{n_tries}\n{'='*40}")
    
    # 1. Get explicit data frames
    train_df, val_df, test_df = get_explicit_splits(
        registry_path="./data/data_registry.csv",
        target_data_type=dataset_type,
        train_config=train_config,
        val_config=val_config,
        seed=seed
    )
    
    # logger.info(f"Split Result -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Split Result -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # 2. Visualize the exact distributions for this try
    try_save_dir = os.path.join(main_directory, f"try_{try_num + 1}")
    os.makedirs(try_save_dir, exist_ok=True)
    
    plot_fold_distributions(
        train_df=train_df, 
        val_df=val_df, 
        test_df=test_df, 
        fold_num=try_num + 1, 
        save_dir=try_save_dir
    )

    # 3. Load the specific tensors needed for this try into RAM
    # (Memory-safe because you only load the 400 needed for train/val right now)
    train_paths = train_df['file_path'].tolist()
    val_paths = val_df['file_path'].tolist()
    
    # -> Load your tensors using the paths
    # -> Run train_model_once_mst()


Starting Try 1/5
Split Result -> Train: 200 | Val: 200 | Test: 1598

Starting Try 2/5
Split Result -> Train: 200 | Val: 200 | Test: 1598

Starting Try 3/5
Split Result -> Train: 200 | Val: 200 | Test: 1598

Starting Try 4/5
Split Result -> Train: 200 | Val: 200 | Test: 1598

Starting Try 5/5
Split Result -> Train: 200 | Val: 200 | Test: 1598
